# MLflow cho Deep Learning, Hugging Face và Transformer

Notebook này tập trung vào ba lớp chủ đề thường đi cùng nhau:
- Deep Learning thuần với vòng lặp train theo epoch.
- Hệ sinh thái Hugging Face cho NLP và Transformer.
- Cách dùng MLflow để theo dõi metric, checkpoint, model và artifact của các workflow đó.

Phạm vi của notebook:
- Minh họa một ví dụ PyTorch cơ bản nhưng có log theo epoch.
- Minh họa cách log Hugging Face pipeline và fine-tuning bằng `Trainer`.
- Giải thích tại sao Transformer khác với MLP/CNN/RNN ở mặt kiến trúc.

Notebook này là phần tiếp nối của `mlflow_ML.ipynb`, không thay thế nó.

## 1. Bức tranh tổng quan

Trong Deep Learning, MLflow thường được dùng để theo dõi:
- loss theo epoch hoặc theo step,
- metric validation như accuracy, F1, BLEU, perplexity,
- checkpoint tốt nhất,
- cấu hình optimizer, learning rate, batch size,
- model cuối cùng hoặc model tốt nhất.

Khác với classical ML, Deep Learning thường có thêm các đặc điểm:
- Huấn luyện kéo dài nhiều epoch.
- Checkpointing và resume training quan trọng hơn.
- GPU/VRAM ảnh hưởng mạnh tới cấu hình chạy.
- Dữ liệu và tokenizer có vai trò lớn trong kết quả cuối cùng.

## 2. Transformer và Hugging Face nằm ở đâu?

Transformer là một họ kiến trúc mạng nơ-ron dựa trên cơ chế `self-attention`. So với RNN/LSTM, Transformer xử lý chuỗi theo cách song song tốt hơn và học quan hệ xa hiệu quả hơn.

Các khối chính của Transformer:
- `Embedding`: biến token thành vector.
- `Positional encoding`: thêm thông tin vị trí vào chuỗi.
- `Self-attention`: cho phép mỗi token nhìn các token khác.
- `Feed-forward block`: biến đổi phi tuyến sau attention.
- `Layer normalization` và `residual connection`: giúp train ổn định hơn.

Hugging Face cung cấp một lớp công cụ thực dụng để làm việc với Transformer:
- `Tokenizer`
- `AutoModel*`
- `pipeline`
- `Trainer`
- `datasets`

Nói ngắn gọn: Transformer là kiến trúc, còn Hugging Face là hệ sinh thái giúp bạn dùng kiến trúc đó hiệu quả hơn.

In [ ]:
# Cài thư viện nếu môi trường chưa có.
# %pip install -q mlflow torch torchvision scikit-learn pandas matplotlib transformers datasets accelerate evaluate sentencepiece


In [ ]:
from pathlib import Path
import json
import tempfile

import mlflow
import mlflow.pytorch
import mlflow.transformers
import numpy as np
import pandas as pd
import torch
from mlflow.models import infer_signature
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

BASE_DIR = Path.cwd()
MLRUNS_DIR = (BASE_DIR / "mlruns").resolve()
MLRUNS_DIR.mkdir(parents=True, exist_ok=True)

tracking_uri = MLRUNS_DIR.as_uri()
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("mlflow_deep_learning_hf")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Tracking URI:", tracking_uri)
print("Device:", device)


## 3. Ví dụ Deep Learning với PyTorch thuần

Ví dụ này mô phỏng một workflow rất điển hình:
- tự viết vòng lặp train,
- log metric sau mỗi epoch,
- lưu checkpoint tốt nhất,
- log model PyTorch vào MLflow.

Đây là pattern phù hợp khi bạn muốn kiểm soát hoàn toàn quá trình huấn luyện.

In [ ]:
X, y = make_classification(
    n_samples=3000,
    n_features=20,
    n_informative=12,
    n_redundant=2,
    n_classes=2,
    random_state=42,
)

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long),
)
val_ds = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long),
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)

class MLPClassifier(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.net(x)

model = MLPClassifier(input_dim=X_train.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
epochs = 8
best_val_loss = float("inf")

with mlflow.start_run(run_name="pytorch_mlp_manual"):
    mlflow.log_params(
        {
            "framework": "pytorch",
            "architecture": "mlp",
            "batch_size": 64,
            "epochs": epochs,
            "learning_rate": 1e-3,
            "hidden_dims": "64-32",
        }
    )

    best_state_dict = None

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_loss_sum = 0.0
        preds, labels = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = loss_fn(logits, yb)
                val_loss_sum += loss.item() * xb.size(0)
                preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                labels.extend(yb.cpu().numpy())

        val_loss = val_loss_sum / len(val_loader.dataset)
        val_acc = accuracy_score(labels, preds)
        val_f1 = f1_score(labels, preds)

        mlflow.log_metrics(
            {
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
                "val_f1": val_f1,
            },
            step=epoch,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    with tempfile.TemporaryDirectory() as tmp_dir:
        checkpoint_path = Path(tmp_dir) / "best_model.pt"
        torch.save(best_state_dict, checkpoint_path)
        mlflow.log_artifact(str(checkpoint_path), artifact_path="checkpoints")

        summary = {
            "best_val_loss": round(float(best_val_loss), 4),
            "last_val_accuracy": round(float(val_acc), 4),
            "last_val_f1": round(float(val_f1), 4),
        }
        summary_path = Path(tmp_dir) / "training_summary.json"
        summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        mlflow.log_artifact(str(summary_path), artifact_path="reports")

    model.load_state_dict(best_state_dict)
    model = model.to("cpu")
    model.eval()
    input_example = X_train[:5].astype("float32")
    with torch.no_grad():
        signature = infer_signature(input_example, model(torch.tensor(input_example)).numpy())

    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
    )


## 4. Nếu muốn ít boilerplate hơn

Với một số workflow PyTorch, bạn có thể thử `mlflow.pytorch.autolog()`. Tuy nhiên với training loop tùy biến, nhiều team vẫn thích log thủ công vì:
- dễ kiểm soát step nào được log,
- dễ gắn thêm artifact đặc thù,
- dễ log checkpoint tốt nhất thay vì checkpoint cuối cùng.

Nói ngắn gọn: càng custom thì log thủ công càng đáng tin.

## 5. Log một Hugging Face pipeline vào MLflow

Đây là cách nhanh nhất để lưu một mô hình Transformer đã có sẵn, ví dụ pipeline phân loại cảm xúc hoặc sinh văn bản.

Tình huống phù hợp:
- Bạn muốn demo nhanh inference.
- Bạn muốn log model pretrained trước khi fine-tune.
- Bạn muốn lưu tokenizer và model trong cùng một artifact MLflow.

In [ ]:
from transformers import pipeline

sentiment_pipe = pipeline(
    task="text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    tokenizer="distilbert-base-uncased-finetuned-sst-2-english",
)

sample_texts = [
    "MLflow makes experiment tracking much easier.",
    "This deployment process is fragile and slow.",
]

with mlflow.start_run(run_name="hf_pipeline_logging"):
    outputs = sentiment_pipe(sample_texts)
    mlflow.log_dict({"inputs": sample_texts, "outputs": outputs}, "examples/pipeline_predictions.json")
    model_info = mlflow.transformers.log_model(
        transformers_model=sentiment_pipe,
        artifact_path="hf_pipeline",
    )

print(model_info.model_uri)


## 6. Fine-tuning Transformer với `Trainer` và MLflow

Khi fine-tune model Hugging Face, cách phổ biến nhất là để `Trainer` report trực tiếp về MLflow thông qua `TrainingArguments(report_to="mlflow")`.

Ý nghĩa thực tế:
- `Trainer` sẽ log loss, learning rate và metric trong quá trình train/eval.
- Bạn vẫn có thể log thêm artifact hoặc model sau khi train xong.
- Đây là workflow tự nhiên nhất nếu team đang dùng Hugging Face làm chuẩn.

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import evaluate

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

raw_ds = load_dataset("imdb")
train_ds = raw_ds["train"].shuffle(seed=42).select(range(1000))
eval_ds = raw_ds["test"].shuffle(seed=42).select(range(200))

def preprocess(batch):
    return tokenizer(batch["text"], truncation=True)

train_ds = train_ds.map(preprocess, batched=True)
eval_ds = eval_ds.map(preprocess, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels)["f1"],
    }

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
args = TrainingArguments(
    output_dir="hf_runs/distilbert_imdb",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,
    report_to="mlflow",
    run_name="distilbert_imdb_ft",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# trainer.train()
# trainer.evaluate()
# mlflow.transformers.log_model(
#     transformers_model={"model": trainer.model, "tokenizer": tokenizer},
#     artifact_path="fine_tuned_transformer",
# )


## 7. Gợi ý mở rộng cho dự án thật

Nếu bạn làm dự án Deep Learning hoặc LLM nghiêm túc hơn, nên cân nhắc log thêm:
- seed, mixed precision mode, gradient accumulation,
- tên GPU, thời gian train, throughput,
- checkpoint tốt nhất, config tokenizer, config model,
- tập dữ liệu xã dự ứng dụng GenAI, prompt engineering, RAG và agent workflow.

Thi liệu chính thức nên đọc thứm:
- MLflow PyTorch docs: `https://mlflow.org/docs/latest/ml/deep-learning/pytorch/`
- MLflow Transformers guide: `https://mlflow.org/docs/latest/ml/deep-learning/transformers/guide/`
